# ECB Shocks x Equity Duration (Macaulay)



Dieses Notebook nutzt folgende Inputs:
- `intermediate/EQDuration_Macaulay.parquet`
- `intermediate/daily_returns_beta.parquet`
- `tables/shocks_ecb_mpd_me_d.csv`

Ziel: Regression von Aktienreaktionen auf ECB-Schocks mit Interaktionen fuer das Macaulay-Duration-Maß.



## 0.) Setup

In [89]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import fdrcorrection

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
TABLE_DIR = BASE_DIR / "tables"

DUR_PATH_MAC = DATA_DIR / "EQDuration_Macaulay.parquet"
DUR_PATH_NP = DATA_DIR / "EQDuration_Netpayout.parquet"
RET_PATH = DATA_DIR / "daily_returns_euro500.parquet"
SHK_PATH = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

for p in [DUR_PATH_MAC, DUR_PATH_NP, RET_PATH, SHK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


## 1.) Helpers

In [90]:

def zscore_by_year(df: pd.DataFrame, col: str, year_col: str = "YEAR") -> pd.Series:
    def _z(s: pd.Series) -> pd.Series:
        mu = s.mean(skipna=True)
        sd = s.std(skipna=True, ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(pd.NA, index=s.index)
        return (s - mu) / sd
    return df.groupby(year_col)[col].transform(_z)


def merge_last_available_feature(
    events: pd.DataFrame,
    features: pd.DataFrame,
    value_col: str,
    event_date_col: str = "date",
    feature_date_col: str = "asof_q_start",
    key_priority=("RIC", "firm_id"),
):
    key = next((k for k in key_priority if k in events.columns and k in features.columns), None)
    if key is None:
        raise ValueError(f"No common key found between events and features (tried {key_priority})")

    left = events.copy()
    left[event_date_col] = pd.to_datetime(left[event_date_col], errors="coerce").dt.normalize()
    left["_row_order"] = np.arange(len(left))

    right = features[[key, feature_date_col, value_col]].copy()
    right[feature_date_col] = pd.to_datetime(right[feature_date_col], errors="coerce").dt.normalize()

    valid_left = left[event_date_col].notna() & left[key].notna()
    valid_right = right[feature_date_col].notna() & right[key].notna() & right[value_col].notna()

    l_ok = left.loc[valid_left].copy()
    r_ok = right.loc[valid_right].copy()

    parts = []
    for k_val, l_grp in l_ok.groupby(key, sort=False):
        r_grp = r_ok.loc[r_ok[key] == k_val]
        if r_grp.empty:
            l_grp = l_grp.copy()
            l_grp[value_col] = np.nan
            parts.append(l_grp)
            continue

        l_grp = l_grp.sort_values(event_date_col)
        r_grp = r_grp.sort_values(feature_date_col)

        m = pd.merge_asof(
            l_grp,
            r_grp,
            left_on=event_date_col,
            right_on=feature_date_col,
            direction="backward",
            allow_exact_matches=True,
        )
        if f"{key}_x" in m.columns:
            m[key] = m[f"{key}_x"]
            m = m.drop(columns=[c for c in [f"{key}_x", f"{key}_y"] if c in m.columns])
        parts.append(m)

    merged_ok = pd.concat(parts, axis=0, ignore_index=False, sort=False) if parts else l_ok.copy()

    left_bad = left.loc[~valid_left].copy()
    out = pd.concat([merged_ok, left_bad], axis=0, ignore_index=False, sort=False)
    out = out.sort_values("_row_order").drop(columns=["_row_order"], errors="ignore")
    out = out.drop(columns=[feature_date_col], errors="ignore")
    return out, key


def _cluster_groups(data: pd.DataFrame, date_col: str, firm_col: str) -> pd.DataFrame:
    d = data.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    if d[date_col].isna().any():
        raise ValueError(f"NaT found in {date_col}")

    d[firm_col] = d[firm_col].astype(str).str.strip()
    if (d[firm_col] == "").any():
        raise ValueError(f"Empty values in {firm_col}")

    return pd.DataFrame(
        {
            "g_date": d[date_col].astype("int64"),
            "g_firm": d[firm_col].astype("category").cat.codes.astype("int64"),
        },
        index=d.index,
    )


def _full_rank_columns(X: pd.DataFrame, tol: float = 1e-12):
    cols = list(X.columns)
    if len(cols) <= 1:
        return cols

    keep = cols.copy()
    while len(keep) > 1:
        mat = X[keep].to_numpy(dtype=float)
        rank = np.linalg.matrix_rank(mat, tol=tol)
        if rank == len(keep):
            break
        variances = X[keep].var(axis=0, skipna=True).fillna(0.0)
        drop_col = variances.idxmin()
        keep.remove(drop_col)
    return keep


def fit_event_fe_2way(
    data: pd.DataFrame,
    y_col: str,
    x_cols: list,
    date_col: str = "date",
    firm_col: str = "firm_id",
    weights=None,
):
    cols = [y_col, date_col, firm_col] + x_cols
    d = data[cols].dropna().copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    if d.empty:
        raise ValueError("Empty regression sample after dropna.")

    for c in [y_col] + x_cols:
        c_dm = f"{c}__dm"
        d[c_dm] = d[c] - d.groupby(date_col)[c].transform("mean")

    y_dm = f"{y_col}__dm"
    x_dm = [f"{c}__dm" for c in x_cols]

    nonzero = []
    for c in x_dm:
        v = pd.to_numeric(d[c], errors="coerce").var(skipna=True)
        if pd.notna(v) and v > 1e-14:
            nonzero.append(c)

    if not nonzero:
        raise ValueError("No regressor variance left after event demeaning.")

    X = d[nonzero].astype(float)
    keep = _full_rank_columns(X)
    X = X[keep]
    y = d[y_dm].astype(float)

    groups = _cluster_groups(d, date_col=date_col, firm_col=firm_col)

    if weights is None:
        m = sm.OLS(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )
    else:
        w = pd.Series(weights, index=data.index).reindex(d.index).astype(float)
        m = sm.WLS(y, X, weights=w).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )

    name_map = {f"{c}__dm": c for c in x_cols}
    m.params.index = [name_map.get(i, i) for i in m.params.index]
    m.bse.index = [name_map.get(i, i) for i in m.bse.index]
    m.tvalues.index = [name_map.get(i, i) for i in m.tvalues.index]
    m.pvalues.index = [name_map.get(i, i) for i in m.pvalues.index]

    return m, d, keep


def apply_fdr(df: pd.DataFrame, p_col: str = "pvalue", term_col: str = "term") -> pd.DataFrame:
    out = df.copy()
    out["p_fdr"] = np.nan
    out["sig_fdr_5pct"] = False
    mask = out[p_col].notna() & out[term_col].str.contains("Duration", na=False)
    if mask.any():
        rej, p_adj = fdrcorrection(out.loc[mask, p_col].to_numpy(), alpha=0.05)
        out.loc[mask, "p_fdr"] = p_adj
        out.loc[mask, "sig_fdr_5pct"] = rej
    return out


def build_interactions(df: pd.DataFrame, dur_std: str, shock_spec: dict, include_mcap: bool = True, include_beta: bool = True):
    x_cols = []

    mp_col = shock_spec.get("mp")
    info_col = shock_spec.get("info")

    if mp_col is not None:
        x_cols.append(f"{mp_col}:{dur_std}")
        if include_beta:
            x_cols.append(f"{mp_col}:BETA_Y_std")
        if include_mcap:
            x_cols.append(f"{mp_col}:MCAP_Y_std")

    if info_col is not None:
        x_cols.append(f"{info_col}:{dur_std}")
        if include_beta:
            x_cols.append(f"{info_col}:BETA_Y_std")
        if include_mcap:
            x_cols.append(f"{info_col}:MCAP_Y_std")

    for c in x_cols:
        a, b = c.split(":")
        df[c] = pd.to_numeric(df[a], errors="coerce") * pd.to_numeric(df[b], errors="coerce")

    return df, x_cols


def assign_duration_bins_with_fallback(df: pd.DataFrame, dur_col: str, year_col: str = "YEAR"):
    d = df.copy()
    d[dur_col] = pd.to_numeric(d[dur_col], errors="coerce")

    stats = d.groupby(year_col)[dur_col].agg(n="count", nunique=lambda s: s.nunique(dropna=True)).reset_index()

    def qbin(s: pd.Series):
        x = pd.to_numeric(s, errors="coerce")
        ok = x.notna()
        out = pd.Series(pd.NA, index=s.index, dtype="object")
        if ok.sum() < 50 or x[ok].nunique() < 5:
            return out
        ranks = x[ok].rank(method="average")
        qcodes = pd.qcut(ranks, q=5, labels=False, duplicates="drop")
        if pd.Series(qcodes).nunique(dropna=True) < 5:
            return out
        labels = pd.Series(qcodes, index=ranks.index).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4", 4: "Q5"})
        out.loc[labels.index] = labels.astype("object")
        return out

    d["Dur_bin"] = d.groupby(year_col)[dur_col].transform(qbin)
    pass_share = float(d["Dur_bin"].isin(["Q1", "Q5"]).mean())

    fallback_used = False
    if pass_share < 0.05:
        fallback_used = True
        x = d[dur_col]
        ranks = x.rank(method="average")
        qcodes = pd.qcut(ranks, q=5, labels=False, duplicates="drop")
        labels = pd.Series(qcodes, index=d.index).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4", 4: "Q5"})
        d["Dur_bin"] = labels.astype("object")

    return d, stats, pass_share, fallback_used

## 2.) Load and clean shock data

In [91]:
df_shk = pd.read_csv(SHK_PATH).copy()
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

# Load both PM and median shock variants for MP and Info
shock_candidates = {
    "MP_pm": ["MP_pm"],
    "MP_median": ["MP_median", "MP_med", "MP_m"],
    "Info_pm": ["CBI_pm", "Info_pm", "INFO_pm"],
    "Info_median": ["CBI_median", "CBI_med", "Info_median", "INFO_median"],
}

shock_map = {}
missing = []
for target, candidates in shock_candidates.items():
    src = next((c for c in candidates if c in df_shk.columns), None)
    if src is None:
        missing.append(f"{target} <- {candidates}")
    else:
        shock_map[src] = target

if missing:
    raise ValueError("Missing required shock columns: " + "; ".join(missing))

df_shk = df_shk.rename(columns=shock_map)
for c in ["MP_pm", "MP_median", "Info_pm", "Info_median"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

SHOCK_COLUMNS = ["MP_pm", "MP_median", "Info_pm", "Info_median"]

df_shk = (
    df_shk[["date"] + SHOCK_COLUMNS]
    .dropna(subset=["date"] + SHOCK_COLUMNS)
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

SHOCK_SPECS = [
    {"name": "TwoShock_pm", "mp": "MP_pm", "info": "Info_pm"},
    {"name": "MP_only_pm", "mp": "MP_pm", "info": None},
    {"name": "Info_only_pm", "mp": None, "info": "Info_pm"},
    {"name": "TwoShock_median", "mp": "MP_median", "info": "Info_median"},
    {"name": "MP_only_median", "mp": "MP_median", "info": None},
    {"name": "Info_only_median", "mp": None, "info": "Info_median"},
]

print("Shock sample:", df_shk.shape)
print("Shock columns:", SHOCK_COLUMNS)
print("Shock specs:", [s["name"] for s in SHOCK_SPECS])
print("corr MP_pm vs Info_pm:", df_shk[["MP_pm", "Info_pm"]].corr().iloc[0, 1])
print("corr MP_median vs Info_median:", df_shk[["MP_median", "Info_median"]].corr().iloc[0, 1])
display(df_shk.head())



Shock sample: (312, 5)
Shock columns: ['MP_pm', 'MP_median', 'Info_pm', 'Info_median']
Shock specs: ['TwoShock_pm', 'MP_only_pm', 'Info_only_pm', 'TwoShock_median', 'MP_only_median', 'Info_only_median']
corr MP_pm vs Info_pm: -0.0016326558000118907
corr MP_median vs Info_median: 0.004767848229290273


,date,MP_pm,MP_median,Info_pm,Info_median
0,1999-01-07,-0.000000,0.020578,-0.037546,-0.058123
1,1999-01-21,0.003581,0.008569,0.000000,-0.004988
2,1999-02-18,-0.000000,-0.005565,-0.000000,0.005565
3,1999-03-04,-0.001926,-0.003596,-0.000000,0.001670
4,1999-03-18,-0.000758,-0.002326,-0.000000,0.001568


## 3.) Load and clean duration panel (firm-quarter)

In [92]:
def prep_duration_quarter_panel(path: Path):
    d = pd.read_parquet(path).copy()

    if "status" in d.columns:
        d = d[d["status"].eq("ok")].copy()

    # Candidate source columns by target duration series
    duration_source_map = {
        "Duration_Macaulay": [
            "Duration_DCF_Macaulay_trim",
            "Duration_DCF_Macaulay",
        ],
        "Duration_Modified": [
            "Duration_Modified_trim",
            "Duration_Modified",
        ],
        "Duration_NetPayout": [
            "Duration_NetPayout_trim",
            "Duration_NetPayout",
            "Duration_NP_trim",
            "Duration_NP",
            "NP_Duration",
        ],
    }

    selected = {}
    for out_col, candidates in duration_source_map.items():
        src = next((c for c in candidates if c in d.columns), None)
        if src is not None:
            d[out_col] = pd.to_numeric(d[src], errors="coerce")
            selected[out_col] = src

    if not selected:
        raise ValueError(f"No duration columns found in {path.name}")

    # Market-cap source from duration panel (quarterly predetermined control)
    mcap_candidates = [
        "mcap_eur", "MCAP_EURO", "marketcap", "market_cap", "mcap", "MarketCap", "market_cap_eur"
    ]
    mcap_source = next((c for c in mcap_candidates if c in d.columns), None)
    if mcap_source is None:
        raise ValueError(
            f"No market-cap column found in {path.name}. "
            f"Expected one of {mcap_candidates}, available columns={list(d.columns)}"
        )
    d["MCAP_Q"] = pd.to_numeric(d[mcap_source], errors="coerce")

    if "RIC" in d.columns:
        d["RIC"] = d["RIC"].astype(str).str.strip()
    if "firm_id" in d.columns:
        d["firm_id"] = d["firm_id"].astype(str).str.strip()

    if "asof_date" in d.columns:
        asof = pd.to_datetime(d["asof_date"], errors="coerce")
    elif "date" in d.columns:
        asof = pd.to_datetime(d["date"], errors="coerce")
    elif "YEAR" in d.columns:
        asof = pd.to_datetime(pd.to_numeric(d["YEAR"], errors="coerce").astype("Int64").astype(str) + "-12-31", errors="coerce")
    elif "year" in d.columns:
        asof = pd.to_datetime(pd.to_numeric(d["year"], errors="coerce").astype("Int64").astype(str) + "-12-31", errors="coerce")
    else:
        raise ValueError("Could not derive date/asof_date for duration panel")

    d["asof_q_start"] = asof.dt.to_period("Q").dt.start_time

    key_cols = [c for c in ["RIC", "firm_id"] if c in d.columns]
    if not key_cols:
        raise ValueError("No key found for duration panel (need RIC or firm_id)")

    keep_value_cols = list(selected.keys()) + ["MCAP_Q"]

    keep_cols = key_cols + ["asof_q_start"] + keep_value_cols
    d = d[keep_cols].dropna(subset=["asof_q_start"]).copy()

    grp_keys = key_cols + ["asof_q_start"]
    d_q = d.groupby(grp_keys, as_index=False)[keep_value_cols].median()

    return d_q, selected, mcap_source


dur_frames = []
dur_source_meta = {}
mcap_source_meta = {}

for label, path in [("Macaulay", DUR_PATH_MAC), ("NetPayout", DUR_PATH_NP)]:
    d_q_i, src_i, mcap_i = prep_duration_quarter_panel(path)
    dur_frames.append(d_q_i)
    dur_source_meta[label] = src_i
    mcap_source_meta[label] = mcap_i

# Combine sources and collapse duplicates on (id, quarter) via median
df_dur_q = pd.concat(dur_frames, ignore_index=True, sort=False)
key_cols = [c for c in ["RIC", "firm_id", "asof_q_start"] if c in df_dur_q.columns]
val_cols = [c for c in ["Duration_Macaulay", "Duration_Modified", "Duration_NetPayout", "MCAP_Q"] if c in df_dur_q.columns]
df_dur_q = df_dur_q.groupby(key_cols, as_index=False)[val_cols].median()

print("Duration panel sample (combined):", df_dur_q.shape)
print("Duration sources by file:", dur_source_meta)
print("MCAP source by file:", mcap_source_meta)
display(df_dur_q.head())




Duration panel sample (combined): (54000, 7)
Duration sources by file: {'Macaulay': {'Duration_Macaulay': 'Duration_DCF_Macaulay_trim', 'Duration_Modified': 'Duration_Modified_trim'}, 'NetPayout': {'Duration_NetPayout': 'Duration_NP'}}
MCAP source by file: {'Macaulay': 'mcap_eur', 'NetPayout': 'mcap_eur'}


,RIC,firm_id,asof_q_start,Duration_Macaulay,Duration_Modified,Duration_NetPayout,MCAP_Q
0,1COV.F,FIRM0001752,2016-01-01,14.893731,13.553296,NaN,6811087500.0
1,1COV.F,FIRM0001752,2016-04-01,15.245462,13.873521,NaN,6674399999.99999
2,1COV.F,FIRM0001752,2016-07-01,14.495485,13.190909,NaN,8088862500.0
3,1COV.F,FIRM0001752,2016-10-01,14.219961,12.940140,NaN,10657574999.999901
4,1COV.F,FIRM0001752,2017-01-01,13.945659,12.690438,NaN,13198950000.0


## 4.) Load and clean daily returns panel (AR + beta source)

In [93]:
df_ret = pd.read_parquet(RET_PATH).copy()

# Harmonize identifiers
if "RIC" not in df_ret.columns:
    for c in ["RIC_current", "ric", "ric_current"]:
        if c in df_ret.columns:
            df_ret = df_ret.rename(columns={c: "RIC"})
            break

if "RIC" not in df_ret.columns:
    raise ValueError("Could not find RIC column in daily returns panel.")

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()

if "firm_id" not in df_ret.columns:
    df_ret["firm_id"] = df_ret["RIC"]

# Harmonize date
if "date" not in df_ret.columns:
    date_candidates = ["Date", "DATE", "trading_date", "TradeDate", "event_date", "day", "datetime"]
    date_src = next((c for c in date_candidates if c in df_ret.columns), None)
    if date_src is None:
        raise ValueError(f"Could not find date column in returns panel. Available columns: {list(df_ret.columns)}")
    df_ret = df_ret.rename(columns={date_src: "date"})

df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
df_ret = df_ret[df_ret["date"].notna()].copy()

# Return source column
ret_candidates = [
    "ret", "return", "returns", "daily_return", "TR.TotalReturn1D", "TR.PricePctChg",
    "RET", "Return", "r", "ret_adj"
]
RET_COL = next((c for c in ret_candidates if c in df_ret.columns), None)

# Ensure AR exists
if "AR" in df_ret.columns:
    df_ret["AR"] = pd.to_numeric(df_ret["AR"], errors="coerce")
    if RET_COL is None:
        RET_COL = "AR"
else:
    if RET_COL is None:
        raise ValueError(
            "Could not find AR or a raw return column to construct AR in daily returns panel."
        )

    df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")
    mkt = df_ret.groupby("date", as_index=False)[RET_COL].mean().rename(columns={RET_COL: "mkt_ret"})
    df_ret = df_ret.merge(mkt, on="date", how="left", validate="m:1")
    df_ret["AR"] = df_ret[RET_COL] - df_ret["mkt_ret"]

# Beta source column
beta_candidates = ["beta", "BETA", "BETA_5Y", "beta_5y", "Beta", "beta_y"]
BETA_COL = next((c for c in beta_candidates if c in df_ret.columns), None)
if BETA_COL is None:
    raise ValueError(
        "Could not find beta column in daily returns panel. "
        f"Tried {beta_candidates}. Available columns: {list(df_ret.columns)}"
    )

df_ret[BETA_COL] = pd.to_numeric(df_ret[BETA_COL], errors="coerce")
df_ret["AR"] = pd.to_numeric(df_ret["AR"], errors="coerce")

# Keep one row per firm-date if accidental duplicates exist
df_ret = (
    df_ret.sort_values(["RIC", "date"])
    .groupby(["RIC", "date"], as_index=False)
    .first()
)

print("Returns panel sample:", df_ret.shape)
print("Return column used:", RET_COL)
print("Beta source column:", BETA_COL)
display(df_ret[["RIC", "firm_id", "date", "AR", BETA_COL]].head())



Returns panel sample: (3327823, 12)
Return column used: ret
Beta source column: beta


,RIC,firm_id,date,AR,beta
0,1COV.F,FIRM0001752,2016-01-04,-0.017727,NaN
1,1COV.F,FIRM0001752,2016-01-05,-0.020493,NaN
2,1COV.F,FIRM0001752,2016-01-06,0.006199,NaN
3,1COV.F,FIRM0001752,2016-01-07,-0.038567,NaN
4,1COV.F,FIRM0001752,2016-01-08,0.019413,NaN


In [94]:
# 4) Build event panel and merge shocks + predetermined durations + predetermined controls

df_evt = df_ret[df_ret["date"].isin(df_shk["date"])].copy()

df_evt = df_evt.merge(
    df_shk[["date"] + SHOCK_COLUMNS],
    on="date",
    how="left",
    validate="m:1",
)

# Predetermined year t-1 (kept for yearly controls)
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# Merge all available duration columns from last available quarter <= event date
merge_keys = {}
for dur_col in [c for c in ["Duration_Macaulay", "Duration_Modified", "Duration_NetPayout"] if c in df_dur_q.columns]:
    df_evt, k = merge_last_available_feature(
        events=df_evt,
        features=df_dur_q,
        value_col=dur_col,
        event_date_col="date",
        feature_date_col="asof_q_start",
        key_priority=("RIC", "firm_id"),
    )
    merge_keys[dur_col] = k

# Merge predetermined market cap from duration panel (quarterly as-of)
if "MCAP_Q" in df_dur_q.columns:
    df_evt, k_mcap = merge_last_available_feature(
        events=df_evt,
        features=df_dur_q,
        value_col="MCAP_Q",
        event_date_col="date",
        feature_date_col="asof_q_start",
        key_priority=("RIC", "firm_id"),
    )
    df_evt = df_evt.rename(columns={"MCAP_Q": "MCAP_Y"})
    merge_keys["MCAP_Y"] = k_mcap
else:
    df_evt["MCAP_Y"] = np.nan

# Build firm-year beta from daily returns, then merge predetermined beta
beta_fy = (
    df_ret.assign(YEAR=pd.to_datetime(df_ret["date"]).dt.year.astype("Int64"))
    [["RIC", "YEAR", BETA_COL]]
    .dropna(subset=["RIC", "YEAR", BETA_COL])
    .groupby(["RIC", "YEAR"], as_index=False)[BETA_COL]
    .median()
    .rename(columns={BETA_COL: "BETA_Y"})
)

df_evt = df_evt.merge(
    beta_fy,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# Central duration specs (single source of truth)
spec_catalog = [
    {"name": "Macaulay", "raw": "Duration_Macaulay"},
    {"name": "Modified", "raw": "Duration_Modified"},
    {"name": "NP_Duration", "raw": "Duration_NetPayout"},
]

DURATION_SPECS_ACTIVE = []
for spec in spec_catalog:
    raw_col = spec["raw"]
    if raw_col in df_evt.columns and df_evt[raw_col].notna().sum() > 0:
        std_col = f"{raw_col}_std"
        df_evt[std_col] = zscore_by_year(df_evt, raw_col, year_col="YEAR")
        DURATION_SPECS_ACTIVE.append({"name": spec["name"], "raw": raw_col, "std": std_col})

if not DURATION_SPECS_ACTIVE:
    raise ValueError("No duration series available in event panel.")

# Predetermined controls (year t-1)
df_evt["BETA_Y_std"] = zscore_by_year(df_evt, "BETA_Y", year_col="YEAR") if "BETA_Y" in df_evt.columns else pd.NA
df_evt["MCAP_Y_std"] = zscore_by_year(df_evt, "MCAP_Y", year_col="YEAR") if "MCAP_Y" in df_evt.columns else pd.NA

print("Return column used:", RET_COL)
print("Beta source column:", BETA_COL)
print("Market cap source columns (duration panel):", mcap_source_meta)
print("Duration merge keys:", merge_keys)
print("Duration specs active:", [s["name"] for s in DURATION_SPECS_ACTIVE])
print("Event panel sample:", df_evt.shape)
display(df_evt.head())



Return column used: ret
Beta source column: beta
Market cap source columns (duration panel): {'Macaulay': 'mcap_eur', 'NetPayout': 'mcap_eur'}
Duration merge keys: {'Duration_Macaulay': 'RIC', 'Duration_Modified': 'RIC', 'Duration_NetPayout': 'RIC', 'MCAP_Y': 'RIC'}
Duration specs active: ['Macaulay', 'Modified', 'NP_Duration']
Event panel sample: (150099, 27)


,date,quarter,name,ISIN,RIC_current,firm_id,ret,market_ret_cap80,beta,mkt_ret,AR,MP_pm,MP_median,Info_pm,Info_median,YEAR,Duration_Macaulay,Duration_Modified,Duration_NetPayout,MCAP_Y,RIC,BETA_Y,Duration_Macaulay_std,Duration_Modified_std,Duration_NetPayout_std,BETA_Y_std,MCAP_Y_std
0,2016-01-21,2016Q1,Covestro AG,DE0006062144,1COVG.F,FIRM0001752,0.05532,0.016671,NaN,0.016006,0.039314,-0.021455,-0.023532,-0.000000,0.002077,2015,14.893731,13.553296,NaN,6811087500.0,1COV.F,NaN,0.095774,0.095789,NaN,NaN,-0.165119
1,2016-03-10,2016Q1,Covestro AG,DE0006062144,1COVG.F,FIRM0001752,-0.00679,-0.014881,NaN,-0.009583,0.002793,0.000000,0.002951,0.049913,0.046962,2015,14.893731,13.553296,NaN,6811087500.0,1COV.F,NaN,0.095774,0.095789,NaN,NaN,-0.165119
2,2016-04-21,2016Q2,Covestro AG,DE0006062144,1COVG.F,FIRM0001752,0.022056,-0.003159,NaN,-0.001856,0.023912,0.001604,0.008748,0.000000,-0.007144,2015,15.245462,13.873521,NaN,6674399999.99999,1COV.F,NaN,0.535439,0.535559,NaN,NaN,-0.173785
3,2016-06-02,2016Q2,Covestro AG,DE0006062144,1COVG.F,FIRM0001752,0.022829,0.002627,NaN,0.002487,0.020342,-0.000000,0.014169,-0.000530,-0.014699,2015,15.245462,13.873521,NaN,6674399999.99999,1COV.F,NaN,0.535439,0.535559,NaN,NaN,-0.173785
4,2016-07-21,2016Q3,Covestro AG,DE0006062144,1COVG.F,FIRM0001752,-0.025849,-0.001904,1.246513,-0.00068,-0.025169,0.000000,0.004206,0.025576,0.021370,2015,14.495485,13.190909,NaN,8088862500.0,1COV.F,NaN,-0.402035,-0.401883,NaN,NaN,-0.084101


## 5.) Baseline Regression

Spezifikation mit Event-FE und 2-way Clustering auf `date` und `firm_id`.



In [95]:
baseline_models = {}
base_tables = []

for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    for shock_spec in SHOCK_SPECS:
        reg_cols = ["AR", "date", "firm_id", dur_std, "BETA_Y_std", "MCAP_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols.append(shock_spec["info"])

        df_reg = df_evt[reg_cols].dropna().copy()
        if df_reg.empty:
            print(f"Skipping baseline ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        df_reg, x_cols = build_interactions(
            df=df_reg,
            dur_std=dur_std,
            shock_spec=shock_spec,
            include_mcap=True,
            include_beta=True,
        )
        if not x_cols:
            print(f"Skipping baseline ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_base, d_base, keep_base = fit_event_fe_2way(
            data=df_reg,
            y_col="AR",
            x_cols=x_cols,
            date_col="date",
            firm_col="firm_id",
        )

        res_base = pd.DataFrame({
            "coef": m_base.params,
            "std_err": m_base.bse,
            "t": m_base.tvalues,
            "pvalue": m_base.pvalues,
        })
        res_base["DurationSpec"] = spec["name"]
        res_base["ShockSpec"] = shock_spec["name"]
        res_base["n_obs"] = len(d_base)

        key = (spec["name"], shock_spec["name"])
        baseline_models[key] = {
            "df_reg": df_reg,
            "x_cols": x_cols,
            "res": res_base,
            "keep": keep_base,
            "sample": d_base,
            "dur_spec": spec,
            "shock_spec": shock_spec,
        }
        base_tables.append(res_base.reset_index().rename(columns={"index": "term"}))

        print(f"Baseline sample ({spec['name']} | {shock_spec['name']}):", d_base.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_base])
        display(res_base)

if base_tables:
    combined_base = pd.concat(base_tables, ignore_index=True)
    combined_base = apply_fdr(combined_base, p_col="pvalue", term_col="term")
    print("Combined baseline table (with FDR on Duration terms):")
    display(combined_base)




Baseline sample (Macaulay | TwoShock_pm): (119569, 16)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std', 'MP_pm:MCAP_Y_std', 'Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std', 'Info_pm:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116236,0.907465,Macaulay,TwoShock_pm,119569
MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388845,0.164880,Macaulay,TwoShock_pm,119569
MP_pm:MCAP_Y_std__dm,-0.008402,0.003252,-2.583672,0.009775,Macaulay,TwoShock_pm,119569
Info_pm:Duration_Macaulay_std__dm,0.011327,0.003497,3.239288,0.001198,Macaulay,TwoShock_pm,119569
Info_pm:BETA_Y_std__dm,0.044776,0.011943,3.749301,0.000177,Macaulay,TwoShock_pm,119569
Info_pm:MCAP_Y_std__dm,0.005145,0.006649,0.773774,0.439065,Macaulay,TwoShock_pm,119569


Baseline sample (Macaulay | MP_only_pm): (119569, 10)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std', 'MP_pm:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116238,0.907464,Macaulay,MP_only_pm,119569
MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388863,0.164875,Macaulay,MP_only_pm,119569
MP_pm:MCAP_Y_std__dm,-0.008402,0.003252,-2.583704,0.009775,Macaulay,MP_only_pm,119569


Baseline sample (Macaulay | Info_only_pm): (119569, 10)
Regressors kept: ['Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std', 'Info_pm:MCAP_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_median:Duration_NetPayout_std__dm,-0.009514,0.003524,-2.699738,0.006939,NP_Duration,MP_only_median,101159
MP_median:BETA_Y_std__dm,-0.037813,0.012572,-3.007818,0.002631,NP_Duration,MP_only_median,101159
MP_median:MCAP_Y_std__dm,-0.013196,0.004302,-3.067191,0.002161,NP_Duration,MP_only_median,101159


Baseline sample (NP_Duration | Info_only_median): (101159, 10)
Regressors kept: ['Info_median:Duration_NetPayout_std', 'Info_median:BETA_Y_std', 'Info_median:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
Info_median:Duration_NetPayout_std__dm,0.008149,0.003787,2.152069,0.031392,NP_Duration,Info_only_median,101159
Info_median:BETA_Y_std__dm,0.056615,0.013237,4.276986,0.000019,NP_Duration,Info_only_median,101159
Info_median:MCAP_Y_std__dm,0.011617,0.005409,2.147747,0.031734,NP_Duration,Info_only_median,101159


Combined baseline table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116236,0.907465,Macaulay,TwoShock_pm,119569,0.907465,False
1,MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388845,0.164880,Macaulay,TwoShock_pm,119569,NaN,False
2,MP_pm:MCAP_Y_std__dm,-0.008402,0.003252,-2.583672,0.009775,Macaulay,TwoShock_pm,119569,NaN,False
3,Info_pm:Duration_Macaulay_std__dm,0.011327,0.003497,3.239288,0.001198,Macaulay,TwoShock_pm,119569,0.007190,True
4,Info_pm:BETA_Y_std__dm,0.044776,0.011943,3.749301,0.000177,Macaulay,TwoShock_pm,119569,NaN,False
...,...,...,...,...,...,...,...,...,...,...
67,MP_median:BETA_Y_std__dm,-0.037813,0.012572,-3.007818,0.002631,NP_Duration,MP_only_median,101159,NaN,False
68,MP_median:MCAP_Y_std__dm,-0.013196,0.004302,-3.067191,0.002161,NP_Duration,MP_only_median,101159,NaN,False
69,Info_median:Duration_NetPayout_std__dm,0.008149,0.003787,2.152069,0.031392,NP_Duration,Info_only_median,101159,0.053815,False
70,Info_median:BETA_Y_std__dm,0.056615,0.013237,4.276986,0.000019,NP_Duration,Info_only_median,101159,NaN,False


## 6.) Robustness 1: Event Equal Weights

Jedes Event bekommt gleiches Gesamtgewicht.



In [96]:
weighted_tables = []

for key, obj in baseline_models.items():
    spec_name, shock_name = key
    df_reg_w = obj["df_reg"].copy()
    x_cols = obj["x_cols"]

    df_reg_w["w_event_equal"] = 1.0 / df_reg_w.groupby("date")["date"].transform("size")
    df_reg_w["w_event_equal"] = df_reg_w["w_event_equal"] / df_reg_w["w_event_equal"].mean()

    m_w, d_w, keep_w = fit_event_fe_2way(
        data=df_reg_w,
        y_col="AR",
        x_cols=x_cols,
        date_col="date",
        firm_col="firm_id",
        weights=df_reg_w["w_event_equal"],
    )

    res_w = pd.DataFrame({
        "coef_w": m_w.params,
        "std_err_w": m_w.bse,
        "t_w": m_w.tvalues,
        "pvalue_w": m_w.pvalues,
    })

    cmp = obj["res"].join(res_w, how="outer")
    cmp["DurationSpec"] = spec_name
    cmp["ShockSpec"] = shock_name
    weighted_tables.append(cmp.reset_index().rename(columns={"index": "term"}))

    print(f"Event-weighted sample ({spec_name} | {shock_name}):", d_w.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_w])
    display(cmp)

if weighted_tables:
    combined_weighted = pd.concat(weighted_tables, ignore_index=True)
    combined_weighted = apply_fdr(combined_weighted, p_col="pvalue_w", term_col="term")
    print("Combined weighted table (with FDR on Duration terms):")
    display(combined_weighted)




Event-weighted sample (Macaulay | TwoShock_pm): (119569, 16)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std', 'MP_pm:MCAP_Y_std', 'Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std', 'Info_pm:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
Info_pm:BETA_Y_std__dm,0.044776,0.011943,3.749301,0.000177,Macaulay,TwoShock_pm,119569,0.048803,0.012089,4.036840,0.000054
Info_pm:Duration_Macaulay_std__dm,0.011327,0.003497,3.239288,0.001198,Macaulay,TwoShock_pm,119569,0.011637,0.004049,2.874230,0.004050
Info_pm:MCAP_Y_std__dm,0.005145,0.006649,0.773774,0.439065,Macaulay,TwoShock_pm,119569,0.004535,0.007228,0.627414,0.530388
MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388845,0.164880,Macaulay,TwoShock_pm,119569,-0.021139,0.014600,-1.447913,0.147641
MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116236,0.907465,Macaulay,TwoShock_pm,119569,0.000281,0.002554,0.110146,0.912293
MP_pm:MCAP_Y_std__dm,-0.008402,0.003252,-2.583672,0.009775,Macaulay,TwoShock_pm,119569,-0.008136,0.003281,-2.479659,0.013151


Event-weighted sample (Macaulay | MP_only_pm): (119569, 10)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std', 'MP_pm:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388863,0.164875,Macaulay,MP_only_pm,119569,-0.021139,0.014599,-1.447931,0.147636
MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116238,0.907464,Macaulay,MP_only_pm,119569,0.000281,0.002554,0.110148,0.912292
MP_pm:MCAP_Y_std__dm,-0.008402,0.003252,-2.583704,0.009775,Macaulay,MP_only_pm,119569,-0.008136,0.003281,-2.479690,0.013150


Event-weighted sample (Macaulay | Info_only_pm): (119569, 10)
Regressors kept: ['Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std', 'Info_pm:MCAP_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
MP_median:BETA_Y_std__dm,-0.037813,0.012572,-3.007818,0.002631,NP_Duration,MP_only_median,101159,-0.034630,0.012530,-2.763864,0.005712
MP_median:Duration_NetPayout_std__dm,-0.009514,0.003524,-2.699738,0.006939,NP_Duration,MP_only_median,101159,-0.009793,0.003719,-2.633413,0.008453
MP_median:MCAP_Y_std__dm,-0.013196,0.004302,-3.067191,0.002161,NP_Duration,MP_only_median,101159,-0.014543,0.005301,-2.743246,0.006084


Event-weighted sample (NP_Duration | Info_only_median): (101159, 10)
Regressors kept: ['Info_median:Duration_NetPayout_std', 'Info_median:BETA_Y_std', 'Info_median:MCAP_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w
Info_median:BETA_Y_std__dm,0.056615,0.013237,4.276986,0.000019,NP_Duration,Info_only_median,101159,0.053630,0.013290,4.035385,0.000055
Info_median:Duration_NetPayout_std__dm,0.008149,0.003787,2.152069,0.031392,NP_Duration,Info_only_median,101159,0.009349,0.004104,2.278150,0.022718
Info_median:MCAP_Y_std__dm,0.011617,0.005409,2.147747,0.031734,NP_Duration,Info_only_median,101159,0.014398,0.006483,2.220714,0.026370


Combined weighted table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,coef_w,std_err_w,t_w,pvalue_w,p_fdr,sig_fdr_5pct
0,Info_pm:BETA_Y_std__dm,0.044776,0.011943,3.749301,0.000177,Macaulay,TwoShock_pm,119569,0.048803,0.012089,4.036840,0.000054,NaN,False
1,Info_pm:Duration_Macaulay_std__dm,0.011327,0.003497,3.239288,0.001198,Macaulay,TwoShock_pm,119569,0.011637,0.004049,2.874230,0.004050,0.019441,True
2,Info_pm:MCAP_Y_std__dm,0.005145,0.006649,0.773774,0.439065,Macaulay,TwoShock_pm,119569,0.004535,0.007228,0.627414,0.530388,NaN,False
3,MP_pm:BETA_Y_std__dm,-0.021158,0.015234,-1.388845,0.164880,Macaulay,TwoShock_pm,119569,-0.021139,0.014600,-1.447913,0.147641,NaN,False
4,MP_pm:Duration_Macaulay_std__dm,0.000299,0.002575,0.116236,0.907465,Macaulay,TwoShock_pm,119569,0.000281,0.002554,0.110146,0.912293,0.979030,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,MP_median:Duration_NetPayout_std__dm,-0.009514,0.003524,-2.699738,0.006939,NP_Duration,MP_only_median,101159,-0.009793,0.003719,-2.633413,0.008453,0.033813,True
68,MP_median:MCAP_Y_std__dm,-0.013196,0.004302,-3.067191,0.002161,NP_Duration,MP_only_median,101159,-0.014543,0.005301,-2.743246,0.006084,NaN,False
69,Info_median:BETA_Y_std__dm,0.056615,0.013237,4.276986,0.000019,NP_Duration,Info_only_median,101159,0.053630,0.013290,4.035385,0.000055,NaN,False
70,Info_median:Duration_NetPayout_std__dm,0.008149,0.003787,2.152069,0.031392,NP_Duration,Info_only_median,101159,0.009349,0.004104,2.278150,0.022718,0.051222,False


## 6.2) Robustness 2: AR[0,+1]


In [97]:
df_ret_01 = df_ret.sort_values(["firm_id", "date"]).copy()
df_ret_01["AR0"] = pd.to_numeric(df_ret_01["AR"], errors="coerce")
df_ret_01["AR1"] = df_ret_01.groupby("firm_id")["AR0"].shift(-1)
df_ret_01["AR_01"] = df_ret_01["AR0"] + df_ret_01["AR1"]

df_evt_01 = df_ret_01[df_ret_01["date"].isin(df_shk["date"])].copy()
df_evt_01 = df_evt_01.merge(df_shk[["date"] + SHOCK_COLUMNS], on="date", how="left", validate="m:1")
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

# merge all available durations from last available quarter <= event date
merge_keys_01 = {}
for spec in DURATION_SPECS_ACTIVE:
    raw_col = spec["raw"]
    df_evt_01, k = merge_last_available_feature(
        events=df_evt_01,
        features=df_dur_q,
        value_col=raw_col,
        event_date_col="date",
        feature_date_col="asof_q_start",
        key_priority=("RIC", "firm_id"),
    )
    merge_keys_01[raw_col] = k

# merge predetermined beta
df_evt_01 = df_evt_01.merge(beta_fy, on=["RIC", "YEAR"], how="left", validate="m:1")

for spec in DURATION_SPECS_ACTIVE:
    raw_col, std_col = spec["raw"], spec["std"]
    if raw_col in df_evt_01.columns:
        df_evt_01[std_col] = zscore_by_year(df_evt_01, raw_col, year_col="YEAR")

if "BETA_Y" in df_evt_01.columns:
    df_evt_01["BETA_Y_std"] = zscore_by_year(df_evt_01, "BETA_Y", year_col="YEAR")

print("Duration merge keys (AR[0,+1]):", merge_keys_01)

res_01_tables = []
for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    for shock_spec in SHOCK_SPECS:
        reg_cols_01 = ["AR_01", "date", "firm_id", dur_std, "BETA_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols_01.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols_01.append(shock_spec["info"])

        df_reg_01 = df_evt_01[reg_cols_01].dropna().copy()
        if df_reg_01.empty:
            print(f"Skipping AR[0,+1] ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        df_reg_01, x_cols_01 = build_interactions(
            df=df_reg_01,
            dur_std=dur_std,
            shock_spec=shock_spec,
            include_mcap=False,
            include_beta=True,
        )
        if not x_cols_01:
            print(f"Skipping AR[0,+1] ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_01, d_01, keep_01 = fit_event_fe_2way(
            data=df_reg_01,
            y_col="AR_01",
            x_cols=x_cols_01,
            date_col="date",
            firm_col="firm_id",
        )

        res_01 = pd.DataFrame({
            "coef": m_01.params,
            "std_err": m_01.bse,
            "t": m_01.tvalues,
            "pvalue": m_01.pvalues,
        })
        res_01["DurationSpec"] = spec["name"]
        res_01["ShockSpec"] = shock_spec["name"]
        res_01["n_obs"] = len(d_01)
        res_01_tables.append(res_01.reset_index().rename(columns={"index": "term"}))

        print(f"AR[0,+1] sample ({spec['name']} | {shock_spec['name']}):", d_01.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_01])
        display(res_01)

if res_01_tables:
    combined_01 = pd.concat(res_01_tables, ignore_index=True)
    combined_01 = apply_fdr(combined_01, p_col="pvalue", term_col="term")
    print("Combined AR[0,+1] table (with FDR on Duration terms):")
    display(combined_01)



Duration merge keys (AR[0,+1]): {'Duration_Macaulay': 'RIC', 'Duration_Modified': 'RIC', 'Duration_NetPayout': 'RIC'}
AR[0,+1] sample (Macaulay | TwoShock_pm): (119569, 12)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std', 'Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_pm:Duration_Macaulay_std__dm,0.004244,0.004372,0.970612,0.331742,Macaulay,TwoShock_pm,119569
MP_pm:BETA_Y_std__dm,-0.014724,0.014959,-0.984310,0.324963,Macaulay,TwoShock_pm,119569
Info_pm:Duration_Macaulay_std__dm,0.009846,0.004729,2.082205,0.037324,Macaulay,TwoShock_pm,119569
Info_pm:BETA_Y_std__dm,0.039168,0.018265,2.144396,0.032001,Macaulay,TwoShock_pm,119569


AR[0,+1] sample (Macaulay | MP_only_pm): (119569, 8)
Regressors kept: ['MP_pm:Duration_Macaulay_std', 'MP_pm:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_pm:Duration_Macaulay_std__dm,0.004244,0.004372,0.970620,0.331738,Macaulay,MP_only_pm,119569
MP_pm:BETA_Y_std__dm,-0.014724,0.014959,-0.984319,0.324959,Macaulay,MP_only_pm,119569


AR[0,+1] sample (Macaulay | Info_only_pm): (119569, 8)
Regressors kept: ['Info_pm:Duration_Macaulay_std', 'Info_pm:BETA_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_median:Duration_NetPayout_std__dm,-0.004954,0.005656,-0.875847,0.381113,NP_Duration,MP_only_median,101159
MP_median:BETA_Y_std__dm,-0.039835,0.012277,-3.244743,0.001176,NP_Duration,MP_only_median,101159


AR[0,+1] sample (NP_Duration | Info_only_median): (101159, 8)
Regressors kept: ['Info_median:Duration_NetPayout_std', 'Info_median:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
Info_median:Duration_NetPayout_std__dm,0.006826,0.005992,1.139270,0.254591,NP_Duration,Info_only_median,101159
Info_median:BETA_Y_std__dm,0.057860,0.016369,3.534622,0.000408,NP_Duration,Info_only_median,101159


Combined AR[0,+1] table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,MP_pm:Duration_Macaulay_std__dm,0.004244,0.004372,0.970612,0.331742,Macaulay,TwoShock_pm,119569,0.455212,False
1,MP_pm:BETA_Y_std__dm,-0.014724,0.014959,-0.984310,0.324963,Macaulay,TwoShock_pm,119569,NaN,False
2,Info_pm:Duration_Macaulay_std__dm,0.009846,0.004729,2.082205,0.037324,Macaulay,TwoShock_pm,119569,0.223943,False
3,Info_pm:BETA_Y_std__dm,0.039168,0.018265,2.144396,0.032001,Macaulay,TwoShock_pm,119569,NaN,False
4,MP_pm:Duration_Macaulay_std__dm,0.004244,0.004372,0.970620,0.331738,Macaulay,MP_only_pm,119569,0.455212,False
5,MP_pm:BETA_Y_std__dm,-0.014724,0.014959,-0.984319,0.324959,Macaulay,MP_only_pm,119569,NaN,False
6,Info_pm:Duration_Macaulay_std__dm,0.009846,0.004729,2.082222,0.037322,Macaulay,Info_only_pm,119569,0.223943,False
7,Info_pm:BETA_Y_std__dm,0.039168,0.018265,2.144414,0.032000,Macaulay,Info_only_pm,119569,NaN,False
8,MP_median:Duration_Macaulay_std__dm,0.004751,0.004983,0.953505,0.340334,Macaulay,TwoShock_median,119569,0.455212,False
9,MP_median:BETA_Y_std__dm,-0.033138,0.012956,-2.557714,0.010536,Macaulay,TwoShock_median,119569,NaN,False


## 6.3) Robustness 3: Portfolio Split (Q1 vs Q5)



In [98]:
# Robustness 3: Portfolio Split (Q1 vs Q5) with diagnostics and fallback

ps_tables = []
for spec in DURATION_SPECS_ACTIVE:
    df_ps = df_evt.copy()
    dur_raw = spec["raw"]

    if dur_raw not in df_ps.columns or df_ps[dur_raw].notna().sum() == 0:
        print(f"Skipping portfolio split ({spec['name']}): no duration data")
        continue

    df_ps, year_diag, pass_share, fallback_used = assign_duration_bins_with_fallback(df_ps, dur_col=dur_raw, year_col="YEAR")
    print(f"Portfolio bin diagnostics ({spec['name']}): pass_share={pass_share:.3f}, fallback_used={fallback_used}")
    display(year_diag.head(10))

    df_ps = df_ps[df_ps["Dur_bin"].isin(["Q1", "Q5"])].copy()
    df_ps["HighDur"] = (df_ps["Dur_bin"] == "Q5").astype(int)

    for shock_spec in SHOCK_SPECS:
        reg_cols_ps = ["AR", "date", "firm_id", "HighDur", "BETA_Y_std"]
        if shock_spec.get("mp") is not None:
            reg_cols_ps.append(shock_spec["mp"])
        if shock_spec.get("info") is not None:
            reg_cols_ps.append(shock_spec["info"])

        df_reg_ps = df_ps[reg_cols_ps].dropna().copy()
        if df_reg_ps.empty:
            print(f"Skipping portfolio split ({spec['name']} | {shock_spec['name']}): empty sample")
            continue

        # Build interactions manually for HighDur specs
        x_cols_ps = []
        mp_col = shock_spec.get("mp")
        info_col = shock_spec.get("info")
        if mp_col is not None:
            x_cols_ps += [f"{mp_col}:HighDur", f"{mp_col}:BETA_Y_std"]
        if info_col is not None:
            x_cols_ps += [f"{info_col}:HighDur", f"{info_col}:BETA_Y_std"]

        for c in x_cols_ps:
            a, b = c.split(":")
            df_reg_ps[c] = pd.to_numeric(df_reg_ps[a], errors="coerce") * pd.to_numeric(df_reg_ps[b], errors="coerce")

        if not x_cols_ps:
            print(f"Skipping portfolio split ({spec['name']} | {shock_spec['name']}): no regressors")
            continue

        m_ps, d_ps, keep_ps = fit_event_fe_2way(
            data=df_reg_ps,
            y_col="AR",
            x_cols=x_cols_ps,
            date_col="date",
            firm_col="firm_id",
        )

        res_ps = pd.DataFrame({
            "coef": m_ps.params,
            "std_err": m_ps.bse,
            "t": m_ps.tvalues,
            "pvalue": m_ps.pvalues,
        })
        res_ps["DurationSpec"] = spec["name"]
        res_ps["ShockSpec"] = shock_spec["name"]
        res_ps["n_obs"] = len(d_ps)
        ps_tables.append(res_ps.reset_index().rename(columns={"index": "term"}))

        print(f"Portfolio split sample ({spec['name']} | {shock_spec['name']}):", d_ps.shape)
        print("Regressors kept:", [k.replace("__dm", "") for k in keep_ps])
        display(res_ps)

if ps_tables:
    combined_ps = pd.concat(ps_tables, ignore_index=True)
    combined_ps = apply_fdr(combined_ps, p_col="pvalue", term_col="term")
    print("Combined portfolio-split table (with FDR on Duration terms):")
    display(combined_ps)




Portfolio bin diagnostics (Macaulay): pass_share=0.356, fallback_used=False


,YEAR,n,nunique
0,1998,7127,1154
1,1999,8612,1333
2,2000,8201,1385
3,2001,4739,1417
4,2002,4828,1437
5,2003,4979,1496
6,2004,5144,1509
7,2005,5233,1584
8,2006,5332,1671
9,2007,5358,1684


Portfolio split sample (Macaulay | TwoShock_pm): (47826, 12)
Regressors kept: ['MP_pm:HighDur', 'MP_pm:BETA_Y_std', 'Info_pm:HighDur', 'Info_pm:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_pm:HighDur__dm,0.002266,0.007221,0.313854,0.753632,Macaulay,TwoShock_pm,47826
MP_pm:BETA_Y_std__dm,-0.025985,0.015512,-1.675133,0.093908,Macaulay,TwoShock_pm,47826
Info_pm:HighDur__dm,0.036838,0.012381,2.975414,0.002926,Macaulay,TwoShock_pm,47826
Info_pm:BETA_Y_std__dm,0.049498,0.015251,3.245549,0.001172,Macaulay,TwoShock_pm,47826


Portfolio split sample (Macaulay | MP_only_pm): (47826, 8)
Regressors kept: ['MP_pm:HighDur', 'MP_pm:BETA_Y_std']


...

,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
MP_median:HighDur__dm,-0.024326,0.010493,-2.318230,0.020437,NP_Duration,MP_only_median,40321
MP_median:BETA_Y_std__dm,-0.045139,0.012040,-3.749057,0.000178,NP_Duration,MP_only_median,40321


Portfolio split sample (NP_Duration | Info_only_median): (40321, 8)
Regressors kept: ['Info_median:HighDur', 'Info_median:BETA_Y_std']


,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs
Info_median:HighDur__dm,0.036506,0.010678,3.418750,0.000629,NP_Duration,Info_only_median,40321
Info_median:BETA_Y_std__dm,0.055856,0.013725,4.069819,0.000047,NP_Duration,Info_only_median,40321


Combined portfolio-split table (with FDR on Duration terms):


,term,coef,std_err,t,pvalue,DurationSpec,ShockSpec,n_obs,p_fdr,sig_fdr_5pct
0,MP_pm:HighDur__dm,0.002266,0.007221,0.313854,0.753632,Macaulay,TwoShock_pm,47826,NaN,False
1,MP_pm:BETA_Y_std__dm,-0.025985,0.015512,-1.675133,0.093908,Macaulay,TwoShock_pm,47826,NaN,False
2,Info_pm:HighDur__dm,0.036838,0.012381,2.975414,0.002926,Macaulay,TwoShock_pm,47826,NaN,False
3,Info_pm:BETA_Y_std__dm,0.049498,0.015251,3.245549,0.001172,Macaulay,TwoShock_pm,47826,NaN,False
4,MP_pm:HighDur__dm,0.002266,0.007221,0.313861,0.753627,Macaulay,MP_only_pm,47826,NaN,False
5,MP_pm:BETA_Y_std__dm,-0.025985,0.015512,-1.675168,0.093901,Macaulay,MP_only_pm,47826,NaN,False
6,Info_pm:HighDur__dm,0.036838,0.012381,2.975476,0.002925,Macaulay,Info_only_pm,47826,NaN,False
7,Info_pm:BETA_Y_std__dm,0.049498,0.015251,3.245617,0.001172,Macaulay,Info_only_pm,47826,NaN,False
8,MP_median:HighDur__dm,0.004166,0.008571,0.486107,0.626891,Macaulay,TwoShock_median,47826,NaN,False
9,MP_median:BETA_Y_std__dm,-0.042563,0.013498,-3.153367,0.001614,Macaulay,TwoShock_median,47826,NaN,False
